# TopAneu A-N00 / A-N00R1 nnU-Net Location

TopAneu Dataset501 기반 뇌동맥류 Location segmentation의
A-N00 baseline과 A-N00R1 corrected retrain 재현 Notebook이다.

실제 Trainer 구현은 repository의 `src/`를 사용하며,
이 Notebook은 데이터 확인, 실행/Resume 및 Final Validation 확인 흐름을 제공한다.

## 1. Google Drive 연결

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. 실행 경로 설정

In [ ]:
from pathlib import Path

REPO = Path(
    "/content/drive/MyDrive/Aneu_MRA/github/model"
)

PREPROCESSED = Path(
    "/content/nnUNet_preprocessed"
)

RESULTS = Path(
    "/content/drive/MyDrive/Aneu_MRA/"
    "nnUNet_results_A_N00R1_sparse53"
)

PYTHON_BIN = Path(
    "/content/venv312/bin/python"
)

print("REPO         :", REPO)
print("PREPROCESSED :", PREPROCESSED)
print("RESULTS      :", RESULTS)
print("PYTHON       :", PYTHON_BIN)

## 3. Dataset501 확인

정상 preprocessed 기준:

- fullres `.b2nd`: 522
- fullres `.pkl`: 261
- GT `.nii.gz`: 261

In [ ]:
dataset = (
    PREPROCESSED
    / "Dataset501_TopAneuMRA_Location"
)

fullres = (
    dataset
    / "nnUNetPlans_3d_fullres"
)

gt_dir = (
    dataset
    / "gt_segmentations"
)

print(
    "b2nd:",
    len(
        list(
            fullres.glob("*.b2nd")
        )
    ),
)

print(
    "pkl :",
    len(
        list(
            fullres.glob("*.pkl")
        )
    ),
)

print(
    "GT  :",
    len(
        list(
            gt_dir.glob("*.nii.gz")
        )
    ),
)

## 4. A-N00R1 실행 / Resume

Launcher가 `checkpoint_latest.pth` 또는 `checkpoint_final.pth`
존재 여부를 확인하여 자동으로 NEW 또는 RESUME을 선택한다.

In [ ]:
import subprocess

launcher = (
    REPO
    / "scripts/training/run_a_n00r1.py"
)

command = [
    str(PYTHON_BIN),
    str(launcher),

    "--preprocessed",
    str(PREPROCESSED),

    "--results",
    str(RESULTS),

    "--python-bin",
    str(PYTHON_BIN),
]

print(
    " ".join(
        command
    )
)

subprocess.run(
    command,
    check=True,
)

## 5. Final Validation 확인

실제 46-patient full-volume validation은
Trainer가 학습 종료 후 `checkpoint_best.pth`로 한 번 수행한다.

In [ ]:
fold_dir = (
    RESULTS
    / "Dataset501_TopAneuMRA_Location"
    / (
        "nnUNetTrainerAneurysmSparse53Retrain"
        "__nnUNetPlans__3d_fullres"
    )
    / "fold_0"
)

validator = (
    REPO
    / "scripts/validation/"
    "check_a_n00r1_final_validation.py"
)

subprocess.run(
    [
        str(PYTHON_BIN),
        str(validator),
        "--fold-dir",
        str(fold_dir),
    ],
    check=False,
)

## 6. 주의사항

- 기존 A-N00 결과 폴더는 수정하지 않는다.
- A-N00R1은 별도의 결과 root를 사용한다.
- checkpoint와 Dataset501 실제 데이터는 GitHub에 올리지 않는다.
- Final Validation 미완료 상태의 중간 결과를 최종 성능으로 확정하지 않는다.